<a href="https://colab.research.google.com/github/vishaljoshi24/DungeonsAndDragonsTurnClassification/blob/Codex-Branch/Experiment_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone -b Codex-Branch https://github.com/vishaljoshi24/DungeonsAndDragonsTurnClassification/
%cd /content/DungeonsAndDragonsTurnClassification

In [ ]:
!pip install dspy

In [ ]:
import argparse

In [ ]:
from modules import  (
    AgentProfile,
    BaselineConversation,
    configure_lm,
    BaselineChatbot
)

In [ ]:
RAT_INFESTATION_SCENARIO = (
'This D&D short scenario specifically concerns a rat infestation. '
'It is set in a craft brewery and is in dire need of help. '
'The players in this scenario are Alice and Bob.'
'The non-player character is Charlie, and he is controlled by the Dungeon Master. '
'Alice and Bob are here to sort out a RAT INFESTATION in the brewery\'s BASEMENT. '
'For the duration of the scenario, only Alice, Bob and the brewery owner Charlie are in the brewery '
'At the beginning of this adventure Alice and Bob '
'meet in the main taproom of the brewery. These two adventurers '
'DO NOT know each other AT FIRST and need to get to know each other. '
'Charlie hands out pints of Ale to Alice and Bob as they get to know each other. '
'Alice and Bob should start by asking Charlie for some information about the infestation. '
'Alice and Bob should then come up with a strategy for clearing out the infestation. '
'Finally, Alice and Bob should explore the brewery\'s cellar based on their strategy. '
'The scenario ends before players start combat. '
)

MISSING_CHILD_SCENARIO = (
  'This D&D short campaign specifically concerns Diana, a magical walrus who has brought a group of animals to safety. '
  'She has brought the animals to the mountains, hoping to protect them from being used by an evil druid to wreak havoc on the local area. '
  'She requests the help of Alice and Bob to stem the threats against everyone. '
  'For the duration of this scenario, only Alice, Bob and Diana are in a cave in the mountains. '
  'At the beginning of this adventure Alice and Bob '
  'meet Diana at her cave in the mountains. The three adventurers '
  'DO NOT know each other AT FIRST and need to get to know each other.'
  'The cave in which the three characters meet has a river which flows out to the open air. '
  'Diana asks Alice and Bob to find her child, an awakened otter that has been spying on a nearby city. '
  'Alice and Bob should start by asking Diana some questions about the missing child. '
  'Alice and Bob should then come up with a strategy for their search of the child. '
  'Finally, Alice and Bob should explore the surrounding mountains based on their strategy. '
)

In [ ]:
def build_conversation() -> BaselineConversation:
    alice = AgentProfile(
        name="Alice",
        character="Alice, a dwarf barbarian.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        f'play the role of a character named {AgentProfile.name} This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One set of common clothes",
                "Two daggers",
                "One axe",
                "50 feet of rope",
                "One tinderbox",
                "One torch",
            ],
        abilities = {""},
        skills = {""},
    )
    bob = AgentProfile(
        name="Bob",
        character="Bob, a wizard.",
        instructions=(
        'The instructions for how to play the role of a D&D player are as '
        'follows. This is a short scenario in which you '
        'play the role of a character named Alice. This scenario '
        'is structured as a Dungeons & Dragons game. '
        'The goal is to be consistent, but creative. It is '
        'important to play the role of a Dungeons & Dragons player as '
        'accurately as possible, i.e., by responding in ways that you think '
        'it is likely a player would respond, and taking '
        'into account all information that you have. '
        'It is important that you collaborate with the  '
        'other player, on the task at hand and listen to the Dungeon Master\'s '
        'insturctions.'
        'Do not be verbose. '
        'Do not repeat yourself. '
        'Always use first-person limited perspective.'
        ),
        inventory = [
                "One Wizard's Staff",
                "One can of oil",
                "One tinderbox",
                "Thunderwave spell: You unleash a wave of thunderous energy.",
                "Command spell: You speak a one-word command to a creature you can see within range."
            ],
        abilities = {""},
        skills = {""},
    )
    dungeon_master = AgentProfile(
          name="Dungeon Master",
          character="The Dungeon Master, who also plays the role of Charlie: the owner of the Wizards Tower Brewing Company.",
          instructions= (
          'This is a tabletop role-playing game: Dungeons & Dragons. You are the Dungeon Master.'
          'You will describe the current situation to the players in the game and then on the basis '
          'of what you tell them they will suggest actions for the character they control. '
          'You will then decide if the action is valid based on Dungeons & Dragons 5th Edition rules. '
          'Aside from you, each other player controls just one character. '
          'If any of the players deviates dramatically from the scenario your response should attempt to re-orient the scenario. '
          'You are the Dungeon Master so you control any non-player characters or adversaries. '
          'You can answer player\'s questions and give them any items they need, but you should '
          'not get involved with the task directly. '
          'You will track the state of the world and keep it consistent as time  '
          'passes in the simulation and the players take actions and change things in their world. '
          'Remember that this is a game. It should be fun for the players. '
          'You should use second-person perspective, when speaking directly to the players. '
          'You should use first-person limited perspective when role-playing as non-player characters and adversaries.'
          'Do not be verbose. Do not repeat yourself.'
          ),
          inventory = [""],
          abilities = {""},
          skills = {""},
    )

    return BaselineConversation(
        agent_a=alice,
        agent_b=bob,
        agent_c=dungeon_master,
        scenario=DEFAULT_SCENARIO,
        chatbot_a=BaselineChatbot(),
        chatbot_b=BaselineChatbot(),
        chatbot_c=BaselineChatbot()
    )

In [ ]:
parser = argparse.ArgumentParser(description="Run three D&D DSPy agents together.")

In [ ]:
parser.add_argument(
    "--opening-message",
    default="You find yourselves in the main taproom of the brewery, questioning Charlie the brewery owner.",
    help="The first turn spoken by the Dungeon Master.",
)

In [ ]:
parser.add_argument(
    "--rounds",
    type=int,
    default=15,
    help="Number of replies to generate after the opening message.",
)

In [ ]:
parser.add_argument("--model", default="ollama_chat/qwen3:8b")

In [ ]:
parser.add_argument("--api-base", default="http://localhost:11434")

In [ ]:
args = parser.parse_args([])

In [ ]:
configure_lm(model=args.model, api_base=args.api_base)

In [ ]:
conversation = build_conversation()

In [ ]:
for turn in conversation.run(args.opening_message, rounds=args.rounds):
    print(turn)

In [ ]:
import dspy
dspy.inspect_history(n=10)